In [0]:
df = (
    spark.read.format('csv')
                .option("header" , True)
                .option("inferSchema" , True)
                .load("/Volumes/ashlamba/bronze/bronze_volume/customers/")
)

In [0]:
df.limit(5).display()

In [0]:
df.printSchema()

In [0]:
df.show(truncate = False)

In [0]:
from pyspark.sql.functions import col,upper
df.select(upper(col("name")).alias('name')).limit(5).display()

In [0]:
from pyspark.sql.functions import expr
df.select(expr("upper(name) as name")).limit(5).display()

In [0]:
df.selectExpr("upper(name) as name").limit(5).display()

In [0]:
from pyspark.sql.functions import col , upper
df = df.withColumn("name" , upper(col("name")))

In [0]:
df.limit(5).display()

In [0]:
from pyspark.sql.functions import col,split
df = df.withColumn("domain" , split(col("email"), "@")[1])

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *
df.groupBy(col("domain")).agg(count("*").alias("total_customer")).orderBy(col("total_customer").desc_nulls_last()).display()

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import current_timestamp 
df = df.withColumn("process_date" , current_timestamp())
df.display()

In [0]:
%sql
create schema ashlamba.silver;

In [0]:
%sql
create schema ashlamba.gold;

In [0]:
from delta.tables import DeltaTable 
from pyspark.sql.functions import expr

if spark.catalog.tableExists("ashlamba.silver.customers_enr"):
    customers_enr_obj = DeltaTable.forName(spark, 'ashlamba.silver.customers_enr')
    (
        customers_enr_obj.alias("t").merge(df.alias("s"), expr("""t.customer_id = s.customer_id"""))
                                        .whenMatchedUpdateAll()
                                        .whenNotMatchedInsertAll()
                                        .execute()
    )
else:
    df.write.format("delta")\
            .mode("append")\
            .saveAsTable("ashlamba.silver.customers_enr")

In [0]:
%sql
drop table ashlamba.silver.customers_enr;

In [0]:
df.count()

In [0]:
%sql
select count(*) from ashlamba.silver.customers_enr;